# CMS Blue Button Test Beneficiary Credentials Update

As of current CMS sandbox testing, BBUser00001 is the only verified working synthetic beneficiary. Use credentials:
- Username: `BBUser00001`
- Password: `PW00001!`

Appropriate scopes must be used or API calls will return "Invalid scopes." errors. Check documentation for correct scope values.

# CMS Blue Button 2.0 API Sample Call

Makes FHIR R4 API calls to retrieve Medicare beneficiary data.

**Inputs:** Credentials and refresh token from `cms_blue_button_api` secret scope.
**Outputs:** FHIR resources (Patient, Coverage, ExplanationOfBenefit) as JSON, with optional Spark DataFrames.

**Workflow:**
1. Run `CMS_Blue_Button_Initial_Setup` once (browser OAuth) to store the initial refresh token
2. Run cells 2-4 to refresh the access token and make API calls
3. Each run of cell 3 automatically updates the stored refresh token
4. Repeat step 2 for each new session

**Note:** CMS uses single-use refresh tokens. Each refresh consumes the old token and issues a new one, which is automatically stored.

In [0]:
import requests
import json

# client_id is a public identifier — not stored in the secret scope
# client_secret is the confidential credential retrieved from the secret scope
client_id = "IyDGHLIUhNe4xblVyCXCnLLzxsB2reD34dDVhrBn"
client_secret = dbutils.secrets.get(scope="cms_blue_button_api", key="client_secret")

# CMS Blue Button 2.0 API base URL
# Sandbox:  https://sandbox.bluebutton.cms.gov
# Production: https://api.bluebutton.cms.gov
BASE_URL = "https://sandbox.bluebutton.cms.gov"

# OAuth redirect URI (must match what is registered with your Blue Button application)
REDIRECT_URI = "http://localhost/callback"

print("Credentials retrieved from secret scope 'cms_blue_button_api'.")
print(f"API base URL: {BASE_URL}")

In [0]:
# Refresh the access token using the stored refresh token
# Run this cell at the start of each session
# CMS Blue Button issues a new refresh token with each refresh (single-use tokens)

from databricks.sdk import WorkspaceClient

refresh_token = dbutils.secrets.get(scope="cms_blue_button_api", key="refresh_token")

refresh_response = requests.post(
    f"{BASE_URL}/v2/o/token/",
    data={
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
        "client_id": client_id,
        "client_secret": client_secret,
    },
)

token_data = refresh_response.json()

if "access_token" in token_data:
    access_token = token_data["access_token"]
    
    # Store the new refresh token (CMS uses single-use refresh tokens)
    if "refresh_token" in token_data:
        w = WorkspaceClient()
        w.secrets.put_secret(
            scope="cms_blue_button_api",
            key="refresh_token",
            string_value=token_data["refresh_token"]
        )
    
    print("✓ Access token refreshed successfully")
    print(f"Valid for: {token_data.get('expires_in')} seconds (1 hour)")
else:
    print("Token refresh failed:")
    print(json.dumps(token_data, indent=2))
    print("\nRe-run the CMS_Blue_Button_Initial_Setup notebook to re-authorize.")

In [0]:
# Make sample FHIR R4 API calls
# The Blue Button API provides access to Patient, Coverage, and
# ExplanationOfBenefit resources in FHIR R4 format.

headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/fhir+json",
}

# --- Patient resource ---
print("=" * 60)
print("Patient Resource")
print("=" * 60)
patient_resp = requests.get(f"{BASE_URL}/v2/fhir/Patient", headers=headers)
patient_data = patient_resp.json()
print(json.dumps(patient_data, indent=2))

# --- Coverage resources ---
print("\n" + "=" * 60)
print("Coverage Resources")
print("=" * 60)
coverage_resp = requests.get(f"{BASE_URL}/v2/fhir/Coverage", headers=headers)
coverage_data = coverage_resp.json()
print(json.dumps(coverage_data, indent=2))

# --- ExplanationOfBenefit resources ---
print("\n" + "=" * 60)
print("ExplanationOfBenefit Resources")
print("=" * 60)
eob_resp = requests.get(f"{BASE_URL}/v2/fhir/ExplanationOfBenefit", headers=headers)
eob_data = eob_resp.json()
print(json.dumps(eob_data, indent=2))

In [0]:
# Optional: Load FHIR resources into Spark DataFrames for further analysis (Expanded)
# This cell flattens the complete FHIR Bundle entries into tabular form.

# Patient DataFrame: expanded fields
patient_entries = patient_data.get("entry", [])
patient_rows = [
    {
        "id": r.get("id", ""),
        "name": r.get("name", [{}])[0].get("text", ""),
        "gender": r.get("gender", ""),
        "birthDate": r.get("birthDate", ""),
        "address": r.get("address", [{}])[0].get("text", ""),
        "telecom": r.get("telecom", [{}])[0].get("value", ""),
        "maritalStatus": r.get("maritalStatus", {}).get("text", ""),
        "identifier": r.get("identifier", [{}])[0].get("value", ""),
        "race": r.get("extension", [{}])[0].get("valueCoding", {}).get("display", ""),
        "ethnicity": r.get("extension", [{}])[1].get("valueCoding", {}).get("display", ""),
        # Add more extension fields as needed
    }
    for r in [e.get("resource", {}) for e in patient_entries]
]
if patient_rows:
    patient_df = spark.createDataFrame(patient_rows)
    display(patient_df)
else:
    print("No patient entries returned.")

# ExplanationOfBenefit DataFrame: expanded fields
eob_entries = eob_data.get("entry", [])
eob_rows = [
    {
        "id": r.get("id", ""),
        "status": r.get("status", ""),
        "created": r.get("created", ""),
        "type": r.get("type", {}).get("coding", [{}])[0].get("code", ""),
        "provider": r.get("provider", {}).get("reference", ""),
        "patient": r.get("patient", {}).get("reference", ""),
        "billablePeriod_start": r.get("billablePeriod", {}).get("start", ""),
        "billablePeriod_end": r.get("billablePeriod", {}).get("end", ""),
        "insurer": r.get("insurer", {}).get("reference", ""),
        "facility": r.get("facility", {}).get("reference", ""),
        "outcome": r.get("outcome", ""),
        "careTeam": r.get("careTeam", [{}])[0].get("provider", {}).get("reference", ""),
        "insurance": r.get("insurance", [{}])[0].get("coverage", {}).get("reference", ""),
        "diagnosis_code": r.get("diagnosis", [{}])[0].get("diagnosisCodeableConcept", {}).get("coding", [{}])[0].get("code", ""),
        "item_service": r.get("item", [{}])[0].get("service", {}).get("coding", [{}])[0].get("code", ""),
        "item_net": r.get("item", [{}])[0].get("net", {}).get("value", ""),
        "total": r.get("total", [{}])[0].get("amount", {}).get("value", ""),
        "payment": r.get("payment", {}).get("amount", {}).get("value", ""),
        "benefitBalance": r.get("benefitBalance", [{}])[0].get("description", ""),
        # Add more nested fields as needed
    }
    for r in [e.get("resource", {}) for e in eob_entries]
]
if eob_rows:
    eob_df = spark.createDataFrame(eob_rows)
    display(eob_df)
else:
    print("No ExplanationOfBenefit entries returned.")